# In-Class Activity: Forecasting Building Electricity Use
In this activity, we will use AI to explore, forecast, and detect anomalies in the hourly electricity use of four campus buildings.

In [ ]:
#@title ▶ Step 0 · Run me first (1–2 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the ✅ line. Untick *load_forecaster* to skip the pretrained forecasting model (Step 2a then has two methods instead of three).
load_forecaster = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp6_tabular_timeseries", "aec_tab"
FOLDERS = ["mp6_tabular_timeseries"]          # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_tab import lab
lab.setup(dataset="workshop", part="series", load_forecaster=load_forecaster)


## Part 1 · Exploring the Time Series
The data are the hourly electricity use (kWh) of four buildings on North American university campuses in 2017, with the outdoor air temperature at each site. Unlike a table, a time series is ordered in time and has patterns that repeat (daily, weekly, and seasonal cycles).

In [ ]:
#@title ▶ Step 1a · Identify the Building Type { display-mode: "form" }
#@markdown The plots show one week in March and the whole year for four unlabeled buildings: an assembly hall, an office, a residence hall, and a school. Use the daily and weekly patterns to decide which building is which.
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
week_title_text = """Building {letter}: one week in March 2017 (Monday–Sunday)"""
year_title_text = """Building {letter}: all of 2017"""
lab.buildings(week_title_text=week_title_text, year_title_text=year_title_text)


In [ ]:
#@title ▶ Step 1b · Check Your Answers { display-mode: "form" }
a = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
b = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
c = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
d = "assembly hall" #@param ["assembly hall", "office", "residence hall", "school"]
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
line_text = """Building {letter}: you answered **{answer}**; it is the **{use}** ({id}, {area} ft², mean {mean} kWh per hour). {mark}"""
total_text = """**{right} of 4** correct."""
lab.buildings_answer(a, b, c, d, line_text=line_text, total_text=total_text)


In [ ]:
#@title ▶ Step 1c · Daily and Weekly Patterns { display-mode: "form" }
#@markdown Choose a building to see its hourly use over the year, one week in March, and its typical day for each weekday (the median of all such days in 2017).
building = "Hog_office_Marlena: office, 131,740 ft²" #@param ["Hog_office_Marlena: office, 131,740 ft²", "Bear_education_Lila: school, 123,610 ft²", "Bear_lodging_Evan: residence hall, 185,200 ft²", "Bear_assembly_Jose: assembly hall, 36,180 ft²"]
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
anatomy_text = """Typical day (the median of each hour over 2017): mean **{mean}** kWh per hour, maximum **{maximum}**, minimum **{minimum}**.
Site air temperature ranged from {temp_min} to {temp_max} °F."""
year_title_text = """{building}: hourly electricity use in 2017 (kWh)"""
week_title_text = """One week in March 2017 (Monday–Sunday)"""
day_title_text = """Typical day: median hourly use for each weekday over 2017"""
lab.anatomy(building, anatomy_text=anatomy_text, year_title_text=year_title_text, week_title_text=week_title_text, day_title_text=day_title_text)


### 📝 Report question 1
> From Steps 1a and 1b: How many buildings did you identify correctly? Which patterns (daily cycle, weekends, seasons) did you use?

> From Step 1c: Choose one building and describe its typical weekday, weekend, and seasonal pattern in three sentences.

## Part 2 · Forecasting Next Week
A forecast predicts future values of a time series from its past. Here, three methods forecast the electricity use for every hour of the week of October 16, 2017 (168 hours), and each forecast is compared with the actual use.

In [ ]:
#@title ▶ Step 2a · Forecast One Week { display-mode: "form" }
#@markdown *same hour last week*: repeats the use from the same hour one week earlier. It is the baseline any model should beat.
#@markdown *decision tree*: gradient-boosted decision trees trained on all data before the test week, with the use one and two weeks earlier, the hour of day, the day of week, and the outdoor temperature as inputs.
#@markdown *a pretrained model (Chronos-Bolt)*: a time-series foundation model trained on many other time series and used zero-shot: it receives only this building's recent history. The shaded area is its 80 % prediction interval.
building = "Hog_office_Marlena: office, 131,740 ft²" #@param ["Hog_office_Marlena: office, 131,740 ft²", "Bear_education_Lila: school, 123,610 ft²", "Bear_lodging_Evan: residence hall, 185,200 ft²", "Bear_assembly_Jose: assembly hall, 36,180 ft²"]
method = "all three" #@param ["all three", "same hour last week", "decision tree", "a pretrained model (Chronos-Bolt)"]
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
title_text = """{building}: forecasts for the week of {week}"""
actual_text = """actual"""
table_text = """Forecast errors over the {hours} hours of the test week:"""
lab.forecast(building, method, title_text=title_text, actual_text=actual_text, table_text=table_text)


### 📝 Report question 2
> From Step 2a: Report the MAE of each method for all four buildings (copy the tables). Which method performs best for each building? Is *same hour last week* ever hard to beat?

> What does the 80 % prediction interval of Chronos-Bolt mean, and how could a facility manager use it?

## Part 3 · Detecting Anomalous Days
An anomaly is a day whose electricity use departs from the building's typical pattern. For each day, the notebook computes the average deviation from the typical use for that weekday and hour, and converts it into a robust z-score: how many robust standard deviations the day is from normal. Days beyond the threshold are flagged.

In [ ]:
#@title ▶ Step 3a · Flag Anomalous Days { display-mode: "form" }
#@markdown Lower the threshold to flag more days; raise it to keep only the most unusual ones. The table lists the flagged days with the US public holidays.
building = "Hog_office_Marlena: office, 131,740 ft²" #@param ["Hog_office_Marlena: office, 131,740 ft²", "Bear_education_Lila: school, 123,610 ft²", "Bear_lodging_Evan: residence hall, 185,200 ft²", "Bear_assembly_Jose: assembly hall, 36,180 ft²"]
threshold = 3.5 #@param {type:"slider", min:2, max:6, step:0.5}
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
title_text = """{building}: daily deviation from the typical weekly pattern ({flagged} days flagged)"""
axis_text = """deviation from typical (kWh per hour)"""
summary_text = """**{flagged} of {days}** days have a robust z-score beyond ±{threshold}; {holidays} of them are public holidays."""
lab.odd_days(building, threshold, title_text=title_text, axis_text=axis_text, summary_text=summary_text)


### 📝 Report question 3
> From Step 3a (two buildings, threshold 3.5): Which flagged days have an obvious cause (see the holiday column), and which do not? For one unexplained day, what would you check first?

> What threshold would you choose for an automatic alert, and why?

## Part 4 · The Same Tasks with Generative/Multimodal AI
Now you will give the same data to **multimodal/generative AI (GPT/HokieAI)**: first to forecast the test week, then to identify anomalous days. The notebook compares its outputs with the methods of Steps 2a and 3a.

In [ ]:
#@title ▶ Step 4a · Forecast with HokieAI { display-mode: "form" }
#@markdown Choose a building and run the cell to reveal the instructions and prompt. HokieAI must return all 168 hourly values; if its output stops early, ask it to continue and paste all parts.
building = "Hog_office_Marlena: office, 131,740 ft²" #@param ["Hog_office_Marlena: office, 131,740 ft²", "Bear_education_Lila: school, 123,610 ft²", "Bear_lodging_Evan: residence hall, 185,200 ft²", "Bear_assembly_Jose: assembly hall, 36,180 ft²"]
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
paste_steps_text = """1. The data is already inside the provided prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy the predictions from HokieAI and paste them into the box below.
4. Click *Score* to evaluate the predictions."""
paste_prompt_text = """Below are two datasets for a building on a North American university campus used as {use}. The first contains the building's hourly electricity use (kWh) and outdoor air temperature (°F) for the four weeks from {start} to {end}. The second contains the outdoor air temperature for the following week, {next_start} to {next_end} (as a weather forecast would provide it).
Forecast the building's electricity use for every hour of that week: {hours} values, from {first_hour} to {last_hour}.
Reply in the following format, one line per hour for all {hours} hours. Use plain numbers without commas:
timestamp, kWh
{first_hour}, ...
{second_hour}, ...
...

Hourly electricity use and temperature:
{history_data}

Next week's temperature:
{temperature_data}"""
button_text = """Score HokieAI predictions"""
reply_text = """Paste HokieAI's full output here."""
plot_text = """{building}: HokieAI and the three methods of Step 2a for the week of {week}"""
compare_text = """Forecast errors over the same hours:"""
note_text = """HokieAI received four weeks of history; the methods of Step 2a were trained on all data before {week}."""
lab.chat_forecast(building, paste_steps_text=paste_steps_text, paste_prompt_text=paste_prompt_text, button_text=button_text, reply_text=reply_text, plot_text=plot_text, compare_text=compare_text, note_text=note_text)


In [ ]:
#@title ▶ Step 4b · Forecast with HokieAI's Data-Analysis Tool { display-mode: "form" }
#@markdown Choose the model HokieAI should train. With its data-analysis tool, HokieAI writes and runs Python code on the attached files instead of predicting from the text alone. If its output shows no code or analysis, ask it to use its data-analysis tool.
building = "Hog_office_Marlena: office, 131,740 ft²" #@param ["Hog_office_Marlena: office, 131,740 ft²", "Bear_education_Lila: school, 123,610 ft²", "Bear_lodging_Evan: residence hall, 185,200 ft²", "Bear_assembly_Jose: assembly hall, 36,180 ft²"]
model = "a decision tree" #@param ["a decision tree", "a random forest", "a straight line", "a small neural network"]
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
steps_text = """1. Download {files} with the buttons below.
2. Open {url} (sign in with your VT account), start a new chat, attach the files, and paste the prompt.
3. Copy the predictions from HokieAI and paste them into the box below.
4. Click *Score* to evaluate the predictions."""
tool_prompt_text = """The attached file {history_file} contains the hourly electricity use (kWh) and outdoor air temperature (°F) of a building on a North American university campus used as {use}, for the four weeks from {start} to {end}. The attached file {temperature_file} contains the outdoor air temperature for the following week, {next_start} to {next_end}.
Use your data-analysis tool (run Python code on the files) to train {model} on {history_file} to predict hourly electricity use. Choose the input features yourself (for example hour of day, day of week, temperature, and the use at the same hour one week earlier). Then forecast every hour of the following week with {temperature_file}: {hours} values, from {first_hour} to {last_hour}.
Reply in the following format, one line per hour for all {hours} hours. Use plain numbers without commas:
timestamp, kWh
{first_hour}, ...
{second_hour}, ...
..."""
button_text = """Score HokieAI predictions"""
reply_text = """Paste HokieAI's full output here."""
plot_text = """{building}: HokieAI and the three methods of Step 2a for the week of {week}"""
compare_text = """Forecast errors over the same hours:"""
note_text = """HokieAI received four weeks of history; the methods of Step 2a were trained on all data before {week}."""
lab.chat_forecast_tool(building, model, steps_text=steps_text, tool_prompt_text=tool_prompt_text, button_text=button_text, reply_text=reply_text, plot_text=plot_text, compare_text=compare_text, note_text=note_text)


In [ ]:
#@title ▶ Step 4c · Identify Anomalous Days with HokieAI { display-mode: "form" }
#@markdown The notebook compares the days HokieAI lists with the days flagged in Step 3a (threshold 3.5) and with the US public holidays.
building = "Hog_office_Marlena: office, 131,740 ft²" #@param ["Hog_office_Marlena: office, 131,740 ft²", "Bear_education_Lila: school, 123,610 ft²", "Bear_lodging_Evan: residence hall, 185,200 ft²", "Bear_assembly_Jose: assembly hall, 36,180 ft²"]
# The wording the notebook shows: edit the text between the triple quotes. Words in {braces} are filled in by the notebook;
# **bold** and *italic* work, and each line is shown as its own line.
paste_steps_text = """1. The data is already inside the provided prompt below.
2. Open {url} (sign in with your VT account), start a new chat, and paste the prompt.
3. Copy the predictions from HokieAI and paste them into the box below.
4. Click *Score* to evaluate the predictions."""
paste_prompt_text = """Below is a dataset for a building on a North American university campus used as {use}: its total daily electricity use (kWh) and mean outdoor air temperature (°F) for every day of {year}.
Identify the anomalous days: days whose electricity use does not fit the building's usual pattern. For each, state whether use was higher or lower than usual and give the most likely reason.
Reply in the following format, one line per day:
date, higher or lower, reason
{year}-01-02, lower, ...

Daily data:
{daily_data}"""
button_text = """Score HokieAI predictions"""
reply_text = """Paste HokieAI's full output here."""
plot_text = """{building}: days flagged by the z-score rule (red) and listed by HokieAI (orange)"""
summary_text = """HokieAI listed **{listed}** days. The z-score rule of Step 3a (threshold {threshold}) flags **{flagged}**: HokieAI found {found}, missed {missed}, and added {extra}."""
holidays_text = """Public holidays in 2017: {holidays}; HokieAI listed {chat_holidays}, the rule flags {rule_holidays}."""
lab.chat_odd_days(building, paste_steps_text=paste_steps_text, paste_prompt_text=paste_prompt_text, button_text=button_text, reply_text=reply_text, plot_text=plot_text, summary_text=summary_text, holidays_text=holidays_text)


### 📝 Report question 4
> From Steps 4a and 4b (one building): Report HokieAI's MAE without and with its data-analysis tool, next to the three methods of Step 2a.

> From Step 4c: How many of the flagged days did HokieAI find, and which days did it add? Check one of its reasons against the data or the calendar: is it plausible?

> Which task suits HokieAI better, forecasting numbers or explaining anomalies, and why?

### Data and model sources
- Electricity meters and weather: Building Data Genome Project 2 (hourly electricity meters and site weather, 2017), Miller et al. (2020), Scientific Data 7:368, MIT, https://github.com/buds-lab/building-data-genome-project-2.
- Pretrained forecasting model: Chronos-Bolt (small), Amazon Science, Apache-2.0, https://huggingface.co/amazon/chronos-bolt-small.
- Generative AI: GPT models via HokieAI (Virginia Tech).